In [59]:
import os
import sys
from pathlib import Path
from earthscope_sfg_workflows.workflows.workflow_handler import WorkflowHandler
from earthscope_sfg_workflows.data_mgmt.model import GARPOSLayout
from earthscope_sfg_workflows.pipelines.config import (
    QCPipelineConfig, QCPinConfig, RinexConfig, PrideConfig, PositionUpdateConfig
)
import logging
from earthscope_sfg_workflows.logging.loggers import set_all_logger_levels
logging.basicConfig(level=logging.INFO)
set_all_logger_levels(logging.INFO)
logger = logging.getLogger(__name__)

SFG Quality Control 
Here, we can demo functions for quick stats over QC data that can serve as a pre-check for more in depth qc processing (i.e. running the full QC pipeline). 



In [60]:
# This folder has the QC tarballs 
qc_directory = Path("/Users/franklyndunbar/Project/SeaFloorGeodesy/PSN011153/NCC1_2025/20250907")

In [61]:
import tarfile
import json
from pathlib import Path
from earthscope_sfg_tools.novatel_tools.rangea_parser import (
    deserialize_rangea,
    extract_rangea_strings_from_qcpin_dict,
    GNSSEpoch
)
import pandas as pd

In [62]:
def epoch_summary(epoch: GNSSEpoch) -> dict:
    """Get summary statistics for an epoch."""
    all_snrs = [
        obs.cn0 
        for sat in epoch.satellites.values() 
        for obs in sat.observations.values()
    ]
    
    systems = {}
    for (system, prn), sat in epoch.satellites.items():
        if system not in systems:
            systems[system] = 0
        systems[system] += 1
    
    return {
        "timestamp": epoch.time,
        "num_satellites": epoch.satellite_count,
        "num_observations": epoch.num_observations,
        "satellites_by_system": systems,
        "avg_snr": sum(all_snrs) / len(all_snrs) if all_snrs else 0,
        "min_snr": min(all_snrs) if all_snrs else None,
        "max_snr": max(all_snrs) if all_snrs else None,
    }

def qc_json_summary(qc_json: dict, logger=logging.getLogger(__name__)) -> dict:
    """Get summary statistics for all epochs in a QC JSON."""
    rangea_strings = extract_rangea_strings_from_qcpin_dict(qc_json)
    range_summaries = []
    
    for rangea_str in rangea_strings:
        epoch = deserialize_rangea(rangea_str)
        range_summaries.append(epoch_summary(epoch))

    # If there are no range summaries, set default values
    if not range_summaries:
        range_summaries = [{"timestamp": None, "num_satellites": 0, "num_observations": 0, "satellites_by_system": {}, "avg_snr": 0, "min_snr": None, "max_snr": None}]
    
    qc_shotdata_summary = {"num_ping_replys": 0, "average_snr": 999, "min_snr": 999, "max_snr": 999}

    return {
        "range_summaries": range_summaries,
        "shotdata_summary": qc_shotdata_summary
    }

In [63]:
def process_qc_tarball(tarball_path: Path) -> dict:    
    with tarfile.open(tarball_path, "r:*") as tar:
            for member in tar.getmembers():
                if member.name.endswith(".pin") and member.isfile():
                    f = tar.extractfile(member)
                    if f is None:
                        continue
                    
                    try:
                        # Read and parse the JSON
                        data = json.load(f)
                    except json.JSONDecodeError as e:
                        logging.error(f"Failed to decode JSON from {member.name} in {tarball_path}: {e}")
                        continue
                
                    summary = qc_json_summary(data)
                    return summary

In [64]:
def build_qc_summary_from_dir(qc_directory: Path) -> dict:
    range_summaries = []
    shotdata_summaries = []
    found_tarballs = 0
    for tarball in qc_directory.glob("*.tar.gz"):
        found_tarballs += 1
        tarball_summary = process_qc_tarball(tarball)
        if tarball_summary is not None:
            range_summaries.extend(tarball_summary.get("range_summaries", []))
            shotdata_summaries.append(tarball_summary.get("shotdata_summary", {}))

    return range_summaries, shotdata_summaries

In [65]:
range_summaries, shotdata_summaries = build_qc_summary_from_dir(qc_directory)

In [66]:
print(range_summaries[0])

{'timestamp': datetime.datetime(2025, 9, 9, 21, 8, 36, 800000, tzinfo=datetime.timezone.utc), 'num_satellites': 17, 'num_observations': 33, 'satellites_by_system': {0: 16, 1: 1}, 'avg_snr': 43.68787878787879, 'min_snr': 37.6, 'max_snr': 51.4}
